In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Specify the file path
file_path = r'C:\Users\vinay\Downloads\CAR_DETAILS.csv'   # Replace with the actual file path

# Try to load the dataset
try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    print(f"File '{file_path}' not found! Please check the file path.")
    exit()  # Exit the program if the file is not found

# Remove leading/trailing whitespaces in column names if the file is loaded
df.columns = df.columns.str.strip()

# Check the column names
print("Columns in the dataset:", df.columns)

# Drop the 'name' column (it contains string values which we do not need for prediction)
df = df.drop(columns=['name'])

# Data preprocessing: Replace missing values with median for numerical columns
numerical_cols = df.select_dtypes(include=['number']).columns  # Select only numerical columns
df[numerical_cols] = df[numerical_cols].fillna(df[numerical_cols].median())

# Splitting the dataset into features (X) and target (y)
X = df.drop(columns=['selling_price'])  # 'selling_price' is the target variable
y = df['selling_price']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define categorical features
categorical_features = ['fuel', 'seller_type', 'transmission', 'owner']

# Ensure categorical features exist in the dataframe
missing_columns = [col for col in categorical_features if col not in df.columns]
if missing_columns:
    print(f"Missing categorical columns: {missing_columns}")
    exit()

# Preprocessing pipeline: Apply OneHotEncoding for categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(), categorical_features)
    ],
    remainder='passthrough'  # Keep the remaining numerical columns as they are
)

# Create a full pipeline: Preprocessing + RandomForestRegressor
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

# Train the model
pipeline.fit(X_train, y_train)

# Predictions
y_pred = pipeline.predict(X_test)

# Evaluate the model using Mean Absolute Error (MAE)
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error: {mae}")

# Optionally: Display the transformed column names after OneHotEncoding
encoded_columns = pipeline.named_steps['preprocessor'].transformers_[0][1].get_feature_names_out()
print("Encoded column names:", encoded_columns)

# Optionally: Display the feature importance of the Random Forest model
feature_importance = pipeline.named_steps['model'].feature_importances_
print("Feature importances:", feature_importance)


Columns in the dataset: Index(['name', 'year', 'selling_price', 'km_driven', 'fuel', 'seller_type',
       'transmission', 'owner'],
      dtype='object')
Mean Absolute Error: 168199.95963999833
Encoded column names: ['fuel_CNG' 'fuel_Diesel' 'fuel_Electric' 'fuel_LPG' 'fuel_Petrol'
 'seller_type_Dealer' 'seller_type_Individual'
 'seller_type_Trustmark Dealer' 'transmission_Automatic'
 'transmission_Manual' 'owner_First Owner' 'owner_Fourth & Above Owner'
 'owner_Second Owner' 'owner_Test Drive Car' 'owner_Third Owner']
Feature importances: [9.29030702e-05 1.19100807e-01 1.72051163e-06 1.83227429e-05
 1.93695694e-02 2.74626450e-02 2.41378988e-02 4.58094992e-03
 1.39749654e-01 1.60506995e-01 9.76284497e-03 3.65568397e-04
 9.73332993e-03 3.98168658e-04 1.96803156e-03 1.98808161e-01
 2.83942429e-01]


In [14]:
from sklearn.metrics import mean_squared_error, r2_score

# Mean Squared Error (MSE)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")

# R-squared (R²) score
r2 = r2_score(y_test, y_pred)
print(f"R-squared: {r2}")


Mean Squared Error: 152994252500.77725
R-squared: 0.49865940598690284


In [15]:
feature_importance = pipeline.named_steps['model'].feature_importances_
for feature, importance in zip(encoded_columns, feature_importance):
    print(f"Feature: {feature}, Importance: {importance}")


Feature: fuel_CNG, Importance: 9.290307018444788e-05
Feature: fuel_Diesel, Importance: 0.11910080689139299
Feature: fuel_Electric, Importance: 1.720511631206504e-06
Feature: fuel_LPG, Importance: 1.8322742863358533e-05
Feature: fuel_Petrol, Importance: 0.019369569430018507
Feature: seller_type_Dealer, Importance: 0.027462645009241594
Feature: seller_type_Individual, Importance: 0.024137898779498984
Feature: seller_type_Trustmark Dealer, Importance: 0.004580949923183329
Feature: transmission_Automatic, Importance: 0.13974965434388067
Feature: transmission_Manual, Importance: 0.1605069953437095
Feature: owner_First Owner, Importance: 0.009762844969560501
Feature: owner_Fourth & Above Owner, Importance: 0.00036556839723442757
Feature: owner_Second Owner, Importance: 0.009733329930560223
Feature: owner_Test Drive Car, Importance: 0.0003981686576187941
Feature: owner_Third Owner, Importance: 0.001968031562899676


In [16]:
import numpy as np
rmse = np.sqrt(mse)
print(f"Root Mean Squared Error: {rmse}")


Root Mean Squared Error: 391144.79735869844


In [17]:
import joblib

# Save the model
joblib.dump(pipeline, 'car_price_predictor_model.pkl')

# Load the model (in the future)
loaded_model = joblib.load('car_price_predictor_model.pkl')


In [18]:
import joblib
joblib.dump(pipeline, 'car_price_predictor.pkl')


['car_price_predictor.pkl']

In [19]:
pip install Flask



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
pip install joblib pandas tk


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import tkinter as tk
from tkinter import messagebox
import joblib
import pandas as pd

# Load the trained model
model = joblib.load('car_price_predictor_model.pkl')

# Function to predict car price based on inputs
def predict_price():
    try:
        # Get the inputs from the user
        year = int(entry_year.get())
        km_driven = int(entry_km_driven.get())
        fuel = fuel_var.get()
        seller_type = seller_type_var.get()
        transmission = transmission_var.get()
        owner = owner_var.get()
        
        # Prepare the input data for prediction
        input_data = {
            'year': year,
            'km_driven': km_driven,
            'fuel': fuel,
            'seller_type': seller_type,
            'transmission': transmission,
            'owner': owner
        }

        # Convert to dataframe for the model
        input_df = pd.DataFrame([input_data])
        
        # Make prediction
        predicted_price = model.predict(input_df)[0]

        # Display the result
        messagebox.showinfo("Prediction", f"Predicted Car Price: ₹{predicted_price:,.2f}")
        
    except Exception as e:
        messagebox.showerror("Error", f"Error in prediction: {e}")

# Initialize the main window
app = tk.Tk()
app.title("Car Price Predictor")
app.geometry("400x350")

# Labels and Inputs
tk.Label(app, text="Year of Manufacture").pack(pady=5)
entry_year = tk.Entry(app)
entry_year.pack(pady=5)

tk.Label(app, text="Kilometers Driven").pack(pady=5)
entry_km_driven = tk.Entry(app)
entry_km_driven.pack(pady=5)

tk.Label(app, text="Fuel Type").pack(pady=5)
fuel_var = tk.StringVar(value="Petrol")
fuel_options = ["Petrol", "Diesel", "CNG", "Electric", "LPG"]
fuel_menu = tk.OptionMenu(app, fuel_var, *fuel_options)
fuel_menu.pack(pady=5)

tk.Label(app, text="Seller Type").pack(pady=5)
seller_type_var = tk.StringVar(value="Dealer")
seller_type_options = ["Dealer", "Individual", "Trustmark Dealer"]
seller_type_menu = tk.OptionMenu(app, seller_type_var, *seller_type_options)
seller_type_menu.pack(pady=5)

tk.Label(app, text="Transmission Type").pack(pady=5)
transmission_var = tk.StringVar(value="Manual")
transmission_options = ["Manual", "Automatic"]
transmission_menu = tk.OptionMenu(app, transmission_var, *transmission_options)
transmission_menu.pack(pady=5)

tk.Label(app, text="Owner Type").pack(pady=5)
owner_var = tk.StringVar(value="First Owner")
owner_options = ["First Owner", "Second Owner", "Third Owner", "Fourth & Above Owner", "Test Drive Car"]
owner_menu = tk.OptionMenu(app, owner_var, *owner_options)
owner_menu.pack(pady=5)

# Predict Button
predict_button = tk.Button(app, text="Predict Price", command=predict_price, bg="blue", fg="white", font="Arial 14")
predict_button.pack(pady=20)

# Run the application
app.mainloop()
